# BÀI TẬP VỀ NHÀ: PROBLEM SET 05 (JUPYTER NOTEBOOK)
## Môn học: Xác suất Thống kê (UET.MAT1052)
### Chủ đề: Nền tảng Xác suất & Tính toán Xác suất với Python

---
**Họ và tên sinh viên:** .....................................................  
**Mã số sinh viên (MSV):** .....................................................  
**Lớp học phần:** .....................................................  

> **Hướng dẫn:** Hoàn thành các phần code (thay thế các vị trí `None` hoặc `# TODO`) và trả lời câu hỏi trực tiếp vào notebook. Chạy kiểm tra tất cả các cell (`Kernel -> Restart & Run All`) trước khi nộp.

### Khởi động thư viện chuẩn

In [ ]:
import numpy as np
import pandas as pd
from plotnine import *

# Cố định random seed để kết quả tái lập tuyệt đối
np.random.seed(2026)
print("Đã nạp đầy đủ thư viện numpy, pandas, plotnine.")

---
### BÀI 1: KHÔNG GIAN MẪU & ĐẠI SỐ BIẾN CỐ MẠNG PHÂN TÁN
Tạo không gian mẫu 9 trạng thái của 2 gateway và kiểm tra số phần tử của các biến cố $A$, $B$, $C$.

In [ ]:
states = ['O', 'C', 'D']
omega = [(g1, g2) for g1 in states for g2 in states]
print(f"Không gian mẫu Omega ({len(omega)} phần tử):\n{omega}")

# TODO: Định nghĩa các tập con biến cố A, B, C bằng list comprehension
event_A = [s for s in omega if s[0] == 'O' and s[1] == 'O']
event_B = [s for s in omega if s[0] == 'D' or s[1] == 'D']
event_C = [s for s in omega if (s[0] == 'C' and s[1] != 'C') or (s[0] != 'C' and s[1] == 'C')]

print("Biến cố A (Cả hai bình thường):", event_A)
print("Biến cố B (Ít nhất 1 sập):", event_B)
print("Biến cố C (Đúng 1 nghẽn):", event_C)

---
### BÀI 2: KHẢO SÁT KỸ NĂNG LẬP TRÌNH (QUY TẮC CỘNG & ĐỘC LẬP THỐNG KÊ)
Khảo sát 200 sinh viên: $n(P) = 140$, $n(C) = 110$, $n(P \cap C) = 70$.

In [ ]:
n_total = 200
n_P = 140
n_C = 110
n_PC = 70

# 1. Xác suất ít nhất 1 ngôn ngữ P(P or C)
prob_union = (n_P + n_C - n_PC) / n_total
print(f"P(P ∪ C) = {prob_union:.4f}")

# 2. Xác suất chỉ biết Python mà không biết C++ P(P \ C)
prob_only_P = (n_P - n_PC) / n_total
print(f"P(P \\ C) = {prob_only_P:.4f}")

# 3. Kiểm tra tính độc lập: P(P and C) vs P(P) * P(C)
prob_P = n_P / n_total
prob_C = n_C / n_total
prob_joint = n_PC / n_total
prob_product = prob_P * prob_C

print(f"P(P ∩ C) = {prob_joint:.4f}")
print(f"P(P) * P(C) = {prob_product:.4f}")
print("Hai biến cố độc lập?", np.isclose(prob_joint, prob_product))

---
### BÀI 3: MÔ PHỎNG MONTE CARLO XÁC SUẤT CÓ ĐIỀU KIỆN (HAI XÚC XẮC)
Mô phỏng gieo 2 xúc xắc độc lập $N = 100,000$ lần để kiểm tra $P(A \mid B)$ và $P(C \mid B)$.

In [ ]:
N = 100_000
dice1 = np.random.randint(1, 7, size=N)
dice2 = np.random.randint(1, 7, size=N)
total = dice1 + dice2

# Biến cố A: Tổng lẻ, B: dice1 == 1, C: Tổng == 7
event_a = (total % 2 == 1)
event_b = (dice1 == 1)
event_c = (total == 7)

# Xác suất tiên nghiệm
p_a_sim = np.mean(event_a)
p_b_sim = np.mean(event_b)
p_c_sim = np.mean(event_c)

# Xác suất có điều kiện P(A|B) và P(C|B)
p_a_given_b_sim = np.mean(event_a[event_b])
p_c_given_b_sim = np.mean(event_c[event_b])

print(f"[Lý thuyết] P(A) = 0.5000, P(C) = 0.1667")
print(f"[Mô phỏng] P(A) = {p_a_sim:.4f}, P(A|B) = {p_a_given_b_sim:.4f}")
print(f"[Mô phỏng] P(C) = {p_c_sim:.4f}, P(C|B) = {p_c_given_b_sim:.4f}")

---
### BÀI 4: PHÂN TÍCH DỮ LIỆU NHÀ MÁY BẰNG CÔNG THỨC TOÀN PHẦN & BAYES
Nạp dữ liệu từ tệp `du-lieu/factory_quality.csv` gồm 1,200 sản phẩm mẫu được ghi nhận từ 3 dây chuyền A, B, C.

In [ ]:
df_factory = pd.read_csv('du-lieu/factory_quality.csv')
print(df_factory.info())
print("\n5 dòng đầu tiên:")
print(df_factory.head())

# Bảng chéo số lượng giữa Dây chuyền và Trạng thái lỗi
ct = pd.crosstab(df_factory['line'], df_factory['status'], margins=True)
print("\nBảng thống kê chéo sản lượng và sản phẩm lỗi:")
print(ct)

In [ ]:
# Tính tỷ lệ lỗi trên mẫu thực tế P(Defective)
p_defective_sample = (df_factory['is_defective'] == 1).mean()
print(f"Tỷ lệ lỗi toàn nhà máy trên tập dữ liệu mẫu: {p_defective_sample:.4f} (Lý thuyết: 0.0360)")

# Lọc riêng các sản phẩm bị lỗi để tính xác suất hậu nghiệm thực tế
df_defects = df_factory[df_factory['is_defective'] == 1]
posterior_sample = df_defects['line'].value_counts(normalize=True).sort_index()
print("\nXác suất hậu nghiệm thực nghiệm P(Dây chuyền | Lỗi):")
print(posterior_sample)

In [ ]:
# Trực quan hóa tỷ lệ sản phẩm lỗi theo dây chuyền bằng plotnine
df_summary = (
    df_factory
    .groupby('line')
    .agg(total_count=('product_id', 'count'), defect_count=('is_defective', 'sum'))
    .assign(defect_rate=lambda d: d['defect_count'] / d['total_count'] * 100)
    .reset_index()
)

p_bar = (
    ggplot(df_summary, aes(x='line', y='defect_rate', fill='line'))
    + geom_col(width=0.6, show_legend=False)
    + geom_text(aes(label='defect_rate.round(2).astype(str) + "%"'), va='bottom', nudge_y=0.2, size=10)
    + scale_fill_manual(values=['#4A90E2', '#50E3C2', '#FF6B6B'])
    + labs(
        title='Tỷ lệ Sản phẩm Lỗi Theo Dây chuyền Sản xuất',
        x='Dây chuyền',
        y='Tỷ lệ lỗi (%)'
    )
    + theme_minimal(base_family='DejaVu Sans')
    + theme(
        plot_title=element_text(face='bold', size=13, hjust=0.5),
        panel_grid_major_x=element_blank()
    )
)
p_bar

---
### BÀI 5: MÔ PHỎNG NGHỊCH LÝ DE MÉRÉ MỞ RỘNG (3 XÚC XẮC)
Mô phỏng $M = 20,000$ ván chơi, mỗi ván gồm 10 lượt gieo 3 xúc xắc độc lập.

In [ ]:
M_games = 20_000
# Ma trận mô phỏng (20,000 ván x 10 lượt x 3 xúc xắc)
sim_dice = np.random.randint(1, 7, size=(M_games, 10, 3))

# Kiểm tra bộ ba đồng nhất: dice1 == dice2 == dice3
triple_match = (sim_dice[:, :, 0] == sim_dice[:, :, 1]) & (sim_dice[:, :, 1] == sim_dice[:, :, 2])

# Người chơi thắng nếu có ít nhất 1 lượt ra bộ ba trong 10 lượt
games_won = np.any(triple_match, axis=1)
win_rate_sim = np.mean(games_won)

theory_win_rate = 1 - (35/36)**10
print(f"Xác suất thắng lý thuyết: {theory_win_rate:.4f} (24.55%)")
print(f"Tỷ lệ thắng mô phỏng ({M_games:,} ván): {win_rate_sim:.4f} ({win_rate_sim*100:.2f}%)")